### <div style="background-color:blue; color:white; padding:10px;"> Imports </div>

In [3]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

### <div style="background-color:blue; color:white; padding:10px;"> Paths and parameters </div>

In [11]:
# ─── Paths and Parameters ────────────────────────────────────────────────────

RGB_ROOT        = r"D:\Datasets\Datasets\EPIC_Kitchen\RGB\P01_04\Original"
FLOW_ROOT       = r"D:\Datasets\Datasets\EPIC_Kitchen\OpticalFlow\P01_04\P01_04"
ANNOTATION_CSV  = r"D:\Datasets\Datasets\EPIC_Kitchen\Label\P01_04.csv"

# EfficientNet-B0 outputs
EFFNET_FEATURES_CSV = "../Features/EPIC/P01_04_effnet_fused_features.csv"
EFFNET_LABELS_CSV   = "../Labels/EPIC/P01_04_effnet_fused_labeled.csv"

# VGG16 outputs
VGG_FEATURES_CSV    = "../Features/EPIC/P01_04_vgg_fused_features.csv"
VGG_LABELS_CSV      = "../Labels/EPIC/P01_04_vgg_fused_labeled.csv"

FEATURE_DIM = 512
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# ─── Load Annotations ────────────────────────────────────────────────────────

annotations = pd.read_csv(ANNOTATION_CSV)
print("Total annotations:", len(annotations))

unique_labels  = sorted(annotations['ActionLabel'].unique())
label_to_idx   = {label: idx for idx, label in enumerate(unique_labels)}
num_classes    = len(unique_labels)
print("Unique ActionLabels:", unique_labels)
print("Label → Index mapping:", label_to_idx)

max_frame = annotations['EndFrame'].max()
labels = np.zeros((max_frame + 1, num_classes), dtype=np.float32)
for _, row in annotations.iterrows():
    start = int(row['StartFrame'])
    end   = int(row['EndFrame'])
    cls   = label_to_idx[int(row['ActionLabel'])]
    labels[start:end + 1, cls] = 1.0
print("Labels shape:", labels.shape)

Using device: cuda
Total annotations: 33
Unique ActionLabels: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28]
Label → Index mapping: {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10, 11: 11, 12: 12, 13: 13, 14: 14, 15: 15, 16: 16, 17: 17, 18: 18, 19: 19, 20: 20, 21: 21, 22: 22, 23: 23, 24: 24, 25: 25, 26: 26, 27: 27, 28: 28}
Labels shape: (6181, 29)


In [12]:
# ─── Image Transform ─────────────────────────────────────────────────────────

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ─── Load Frame Paths ─────────────────────────────────────────────────────────

rgb_frames  = sorted(glob.glob(os.path.join(RGB_ROOT,  "*.jpg")))
flow_frames = sorted(glob.glob(os.path.join(FLOW_ROOT, "*.jpg")))
assert len(rgb_frames) == len(flow_frames), "Mismatch in RGB and Flow frames"
num_frames = len(rgb_frames)
print("Total frames:", num_frames)

# ─── Adaptive Fusion Module ──────────────────────────────────────────────────

class AdaptiveFusion(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.fc      = nn.Linear(dim * 2, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, rgb, flow):
        x     = torch.cat([rgb, flow], dim=1)
        alpha = self.sigmoid(self.fc(x))
        fused = alpha * rgb + (1 - alpha) * flow
        return fused

Total frames: 6180


In [13]:
# ─── Generic Feature Extraction Pipeline ─────────────────────────────────────

def build_extractor_effnetb0(feature_dim, device):
    """
    EfficientNet-B0:
      - Backbone output dim : 1280 (after global avg pool)
      - Projection          : 1280 → feature_dim
    """
    backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    # Keep everything except the final classifier
    # backbone.classifier is Sequential(Dropout, Linear)
    backbone.classifier = nn.Identity()          # output: (B, 1280)
    projection = nn.Linear(1280, feature_dim)
    backbone   = backbone.to(device).eval()
    projection = projection.to(device).eval()
    return backbone, projection


def build_extractor_vgg16(feature_dim, device):
    """
    VGG16:
      - Backbone output dim : 512 spatial maps → AdaptiveAvgPool → 512
      - Projection          : 512 → feature_dim
    """
    backbone = models.vgg16(weights=models.VGG16_Weights.DEFAULT)
    # VGG16: features (conv layers) → avgpool → classifier
    # We use only the conv feature extractor + global avg pool
    feature_extractor = nn.Sequential(
        backbone.features,                        # output: (B, 512, 7, 7)
        nn.AdaptiveAvgPool2d((1, 1)),             # output: (B, 512, 1, 1)
        nn.Flatten()                              # output: (B, 512)
    )
    projection = nn.Linear(512, feature_dim)
    feature_extractor = feature_extractor.to(device).eval()
    projection        = projection.to(device).eval()
    return feature_extractor, projection


@torch.no_grad()
def extract_feature(image_path, backbone, projection, transform, device):
    image   = Image.open(image_path).convert("RGB")
    image   = transform(image).unsqueeze(0).to(device)
    feature = backbone(image)                     # (1, backbone_dim)
    feature = feature.view(1, -1)
    feature = projection(feature)                 # (1, FEATURE_DIM)
    return feature.squeeze(0).cpu().numpy()


def extract_all_features(rgb_frames, flow_frames, backbone, projection,
                          transform, device, feature_dim, desc=""):
    num_frames    = len(rgb_frames)
    rgb_features  = np.zeros((num_frames, feature_dim), dtype=np.float32)
    flow_features = np.zeros((num_frames, feature_dim), dtype=np.float32)

    for i in tqdm(range(num_frames), desc=desc):
        rgb_features[i]  = extract_feature(rgb_frames[i],  backbone, projection, transform, device)
        flow_features[i] = extract_feature(flow_frames[i], backbone, projection, transform, device)

    print(f"  RGB  features : {rgb_features.shape}")
    print(f"  Flow features : {flow_features.shape}")
    return rgb_features, flow_features


def fuse_and_save(rgb_features, flow_features, labels, num_frames,
                  features_csv, labels_csv, device, feature_dim):
    # Adaptive fusion
    fusion_model = AdaptiveFusion(feature_dim).to(device)
    fusion_model.eval()

    rgb_tensor  = torch.from_numpy(rgb_features).to(device)
    flow_tensor = torch.from_numpy(flow_features).to(device)

    with torch.no_grad():
        fused_tensor = fusion_model(rgb_tensor, flow_tensor)

    fused_features = fused_tensor.cpu().numpy()
    print(f"  Fused features shape: {fused_features.shape}")

    # Align labels
    lbl = labels.copy()
    if lbl.shape[0] < num_frames:
        padding = np.zeros((num_frames - lbl.shape[0], lbl.shape[1]), dtype=np.float32)
        lbl = np.vstack([lbl, padding])
    elif lbl.shape[0] > num_frames:
        lbl = lbl[:num_frames]

    # Save features
    os.makedirs(os.path.dirname(features_csv), exist_ok=True)
    features_df = pd.DataFrame(fused_features)
    features_df.insert(0, "frame_id", np.arange(num_frames))
    features_df.to_csv(features_csv, index=False)
    print(f"  Saved features → {features_csv}")

    # Save labels
    os.makedirs(os.path.dirname(labels_csv), exist_ok=True)
    labels_df = pd.DataFrame(lbl)
    labels_df.insert(0, "frame_id", np.arange(num_frames))
    labels_df.to_csv(labels_csv, index=False)
    print(f"  Saved labels   → {labels_csv}")


# ═══════════════════════════════════════════════════════════════════════════════
# 1️⃣  EfficientNet-B0
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("  EfficientNet-B0 Feature Extraction")
print("="*60)

eff_backbone, eff_projection = build_extractor_effnetb0(FEATURE_DIM, DEVICE)

rgb_feat_eff, flow_feat_eff = extract_all_features(
    rgb_frames, flow_frames,
    eff_backbone, eff_projection,
    transform, DEVICE, FEATURE_DIM,
    desc="EfficientNet-B0"
)

fuse_and_save(
    rgb_feat_eff, flow_feat_eff, labels, num_frames,
    EFFNET_FEATURES_CSV, EFFNET_LABELS_CSV,
    DEVICE, FEATURE_DIM
)

# ═══════════════════════════════════════════════════════════════════════════════
# 2️⃣  VGG16
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("  VGG16 Feature Extraction")
print("="*60)

vgg_backbone, vgg_projection = build_extractor_vgg16(FEATURE_DIM, DEVICE)

rgb_feat_vgg, flow_feat_vgg = extract_all_features(
    rgb_frames, flow_frames,
    vgg_backbone, vgg_projection,
    transform, DEVICE, FEATURE_DIM,
    desc="VGG16"
)

fuse_and_save(
    rgb_feat_vgg, flow_feat_vgg, labels, num_frames,
    VGG_FEATURES_CSV, VGG_LABELS_CSV,
    DEVICE, FEATURE_DIM
)

print("\n✅ All feature extractions complete!")


  EfficientNet-B0 Feature Extraction


EfficientNet-B0: 100%|█████████████████████████████████████████████████████████████| 6180/6180 [08:17<00:00, 12.43it/s]


  RGB  features : (6180, 512)
  Flow features : (6180, 512)
  Fused features shape: (6180, 512)
  Saved features → ../Features/EPIC/P01_04_effnet_fused_features.csv
  Saved labels   → ../Labels/EPIC/P01_04_effnet_fused_labeled.csv

  VGG16 Feature Extraction


VGG16: 100%|███████████████████████████████████████████████████████████████████████| 6180/6180 [04:19<00:00, 23.83it/s]


  RGB  features : (6180, 512)
  Flow features : (6180, 512)
  Fused features shape: (6180, 512)
  Saved features → ../Features/EPIC/P01_04_vgg_fused_features.csv
  Saved labels   → ../Labels/EPIC/P01_04_vgg_fused_labeled.csv

✅ All feature extractions complete!
